# Score calculations for all six embeddings

## LogME score

In [1]:
import os
import json
import numpy as np

In [3]:
def compute_logme(features: np.ndarray, labels: np.ndarray, max_iter: int = 25, tol: float = 1e-4):
    """
    Computes overall and per-class LogME scores with stable MacKay fixed-point updates.
    """
    N, D = features.shape
    unique_classes = np.unique(labels)
    num_classes = len(unique_classes)
    
    # 1. Mean-center features
    F = features - np.mean(features, axis=0, keepdims=True)
    
    # 2. Thin SVD: F = U * diag(s) * V^T
    U, s, _ = np.linalg.svd(F, full_matrices=False)
    k = len(s)
    s2 = s ** 2

    class_scores = {}
    total_log_evidence = 0.0

    for c in unique_classes:
        # Binary target centered to remove DC bias
        y = (labels == c).astype(np.float64)
        y_centered = y - np.mean(y)
        
        y_proj = U.T @ y_centered          # Shape: (k,)
        y_norm_sq = np.dot(y_centered, y_centered)
        y_proj_sq = np.sum(y_proj ** 2)
        y_ortho_sq = max(0.0, y_norm_sq - y_proj_sq)  # Energy outside feature span

        # Initialize hyperparameters
        alpha = 1.0
        beta = 1.0 / (np.var(y_centered) + 1e-4)

        for _ in range(max_iter):
            prev_alpha, prev_beta = alpha, beta
            
            # Denominator: alpha + beta * s_i^2
            denom = alpha + beta * s2
            
            # Effective degrees of freedom
            gamma = np.sum((beta * s2) / denom)
            
            # Posterior mean weights (including beta)
            m = (beta * s / denom) * y_proj
            w_norm_sq = np.sum(m ** 2)
            
            # Exact RSS: orthogonal part + regularized projection error
            rss = y_ortho_sq + np.sum(((alpha / denom) * y_proj) ** 2)

            # Fixed-point parameter updates
            alpha = gamma / (w_norm_sq + 1e-12)
            beta = (N - gamma) / (rss + 1e-12)

            if abs(alpha - prev_alpha) / (alpha + 1e-12) < tol and abs(beta - prev_beta) / (beta + 1e-12) < tol:
                break

        # Log determinant: log |C|
        log_det = -N * np.log(beta) - k * np.log(alpha) + np.sum(np.log(alpha + beta * s2))
        
        # Quadratic form: y^T C^{-1} y
        quad_form = beta * y_ortho_sq + np.sum((alpha * beta / (alpha + beta * s2)) * (y_proj ** 2))
        
        # Marginal log likelihood for class c
        log_evidence_c = -0.5 * (N * np.log(2.0 * np.pi) + log_det + quad_form)
        
        # Normalized score per sample
        class_score = float(log_evidence_c / N)
        class_scores[int(c)] = class_score
        total_log_evidence += log_evidence_c

    overall_score = float(total_log_evidence / (N * num_classes))
    return overall_score, class_scores

# --- Execution Pipeline ---
base_dir = "./model_embeddings"
output_dir = "./model_scores/logME"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating {model_name}...")
    data = np.load(npz_path)
    features = data["embeddings"]
    labels = data["labels"]
    
    overall, class_wise = compute_logme(features, labels)
    
    results[model_name] = {
        "overall_logme": overall,
        "class_wise_logme": class_wise
    }

output_file = os.path.join(output_dir, "all_models_logme.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved updated LogME scores to {output_file}")

Evaluating AVES-bio...
Evaluating BirdAVES-biox-base...
Evaluating DistilHuBERT...
Evaluating HuBERT...
Evaluating MusicHuBERT...
Evaluating WavLM...

Saved updated LogME scores to ./model_scores/logME\all_models_logme.json


## H score

In [4]:
import os
import json
import numpy as np

In [7]:

def compute_hscore(features: np.ndarray, labels: np.ndarray, eps: float = 1e-4):
    """
    Computes overall and per-class H-scores for features (N, D) and labels (N,).
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    
    # 1. Mean-centre features across samples
    F_centered = features - np.mean(features, axis=0, keepdims=True)

    # 2. Total feature covariance matrix with Tikhonov regularisation
    Sigma_f = (F_centered.T @ F_centered) / N
    Sigma_f_reg = Sigma_f + eps * np.eye(D)
    Sigma_f_inv = np.linalg.pinv(Sigma_f_reg)

    # 3. Class-wise centroid Mahalanobis evaluation
    class_scores = {}
    overall_hscore = 0.0

    for c, count in zip(unique_classes, counts):
        idx = np.where(labels == c)[0]
        mu_c = np.mean(F_centered[idx], axis=0)  # Class-conditional mean vector (D,)
        
        # Squared Mahalanobis distance of class centroid
        h_c = float(mu_c.T @ Sigma_f_inv @ mu_c)
        class_scores[int(c)] = h_c
        
        # Prior-weighted contribution to the overall trace
        overall_hscore += (count / N) * h_c

    return float(overall_hscore), class_scores

# --- Execution Pipeline ---
base_dir = "./model_embeddings"
output_dir = "./model_scores/H_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating H-score for {model_name}...")
    data = np.load(npz_path)
    features = data["embeddings"]
    labels = data["labels"]
    
    overall, class_wise = compute_hscore(features, labels)
    
    results[model_name] = {
        "overall_hscore": overall,
        "class_wise_hscore": class_wise
    }

output_file = os.path.join(output_dir, "all_models_hscore.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved all H-scores to {output_file}")

Evaluating H-score for AVES-bio...
Evaluating H-score for BirdAVES-biox-base...
Evaluating H-score for DistilHuBERT...
Evaluating H-score for HuBERT...
Evaluating H-score for MusicHuBERT...
Evaluating H-score for WavLM...

Saved all H-scores to ./model_scores/H_score\all_models_hscore.json


# NTK - LFC

In [1]:
import os
import json
import numpy as np

def compute_ntk_lfc(features: np.ndarray, labels: np.ndarray, kernel_type: str = "linear"):
    """
    Computes overall and per-class NTK-based Label-Feature Correlation (LFC).
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    
    # 1. L2 Normalize features (project onto unit hypersphere)
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    F_norm = features / norms

    # 2. Compute empirical feature kernel matrix (N x N)
    cos_sim = np.clip(F_norm @ F_norm.T, -1.0, 1.0)
    
    if kernel_type == "linear":
        K = cos_sim
    elif kernel_type == "relu_ntk":
        theta = np.arccos(cos_sim)
        K = (1.0 / np.pi) * (np.sin(theta) + (np.pi - theta) * cos_sim)
    else:
        raise ValueError("Unsupported kernel type. Use 'linear' or 'relu_ntk'.")

    # Frobenius norm of feature kernel
    norm_K = np.linalg.norm(K, 'fro')

    # 3. Calculate class-wise and overall alignment
    numerator_total = 0.0
    sum_nc_sq = 0.0
    class_scores = {}

    for c, n_c in zip(unique_classes, counts):
        idx = np.where(labels == c)[0]
        
        # Extract submatrix of pairwise affinities within class c
        K_c = K[np.ix_(idx, idx)]
        class_intra_affinity = float(np.sum(K_c))
        
        # Individual class alignment score
        class_scores[int(c)] = class_intra_affinity / (norm_K * n_c)
        
        # Accumulate for overall score
        numerator_total += class_intra_affinity
        sum_nc_sq += float(n_c ** 2)

    # Frobenius norm of ideal label kernel
    norm_KY = np.sqrt(sum_nc_sq)
    overall_lfc = float(numerator_total / (norm_K * norm_KY))

    return overall_lfc, class_scores

# --- Execution Pipeline ---
base_dir = "./model_embeddings"
output_dir = "./model_scores/LFC_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating LFC score for {model_name}...")
    data = np.load(npz_path)
    features = data["embeddings"]
    labels = data["labels"]
    
    overall, class_wise = compute_ntk_lfc(features, labels, kernel_type="linear")
    
    results[model_name] = {
        "overall_lfc": overall,
        "class_wise_lfc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lfc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved all LFC scores to {output_file}")

Evaluating LFC score for AVES-bio...
Evaluating LFC score for BirdAVES-biox-base...
Evaluating LFC score for DistilHuBERT...
Evaluating LFC score for HuBERT...
Evaluating LFC score for MusicHuBERT...
Evaluating LFC score for WavLM...

Saved all LFC scores to ./model_scores/LFC_score\all_models_lfc.json


In [2]:
import os
import json
import numpy as np

def compute_ntk_lfc(features: np.ndarray, labels: np.ndarray, centered: bool = True):
    """
    Computes overall and class-wise NTK LFC for N=150, D=768/1024, C=6 (N_c=25).
    
    Args:
        features: (N, D) array of embeddings
        labels: (N,) array of integer labels
        centered: If True, centers the kernel matrix (removes ~0.408 anisotropy floor)
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    num_classes = len(unique_classes)
    
    # 1. Unit-norm normalize embeddings
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    F_norm = features / norms

    # 2. Linear NTK / Gram Matrix
    K = np.clip(F_norm @ F_norm.T, -1.0, 1.0)
    
    # Construct ideal block-diagonal label kernel K_Y
    Y_onehot = np.eye(num_classes)[labels]  # (N, C)
    K_Y = Y_onehot @ Y_onehot.T             # (N, N)

    # 3. Kernel Centering: H * K * H, where H = I - (1/N) 1 1^T
    if centered:
        H = np.eye(N) - (1.0 / N) * np.ones((N, N))
        K = H @ K @ H
        K_Y = H @ K_Y @ H

    norm_K = np.linalg.norm(K, 'fro') + 1e-12
    norm_KY = np.linalg.norm(K_Y, 'fro') + 1e-12

    # 4. Class-wise decomposition
    class_scores = {}
    total_alignment = 0.0

    for c in unique_classes:
        y_c = (labels == c).astype(np.float64)
        
        if centered:
            # Centered target vector
            y_c_tilde = y_c - np.mean(y_c)
            # Quadratic form evaluates the inner product for class c
            intra_c = float(y_c_tilde.T @ K @ y_c_tilde)
        else:
            idx = np.where(labels == c)[0]
            intra_c = float(np.sum(K[np.ix_(idx, idx)]))

        # Consistent partition: each class accounts for its share of the total alignment
        score_c = intra_c / (norm_K * norm_KY)
        class_scores[int(c)] = score_c
        total_alignment += score_c

    # Overall LFC equals the sum of class-wise contributions
    overall_lfc = float(total_alignment)

    return overall_lfc, class_scores

# --- Execution Pipeline ---
base_dir = "./model_embeddings"
output_dir = "./model_scores/LFC_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating LFC for {model_name}...")
    data = np.load(npz_path)
    features = data["embeddings"]
    labels = data["labels"]
    
    overall, class_wise = compute_ntk_lfc(features, labels, centered=True)
    
    results[model_name] = {
        "overall_lfc": overall,
        "class_wise_lfc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lfc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved centered LFC scores to {output_file}")

Evaluating LFC for AVES-bio...
Evaluating LFC for BirdAVES-biox-base...
Evaluating LFC for DistilHuBERT...
Evaluating LFC for HuBERT...
Evaluating LFC for MusicHuBERT...
Evaluating LFC for WavLM...

Saved centered LFC scores to ./model_scores/LFC_score\all_models_lfc.json
